# 🚀 Next-Candle Prediction System: Multi-Timeframe Deep Learning Ensemble

**Version:** 2.0 (2025 State-of-the-Art Edition)  
**Created:** February 2026  
**File:** `Next_Candle_Prediction_MultiTF_Pro_v2.ipynb`

This notebook is a production-style research template for BTC/USD (or any symbol) next-candle **direction prediction** using:
- Multi-timeframe market data
- Advanced feature engineering
- Tree models + deep learning models
- Stacking ensemble
- Explainability (SHAP)
- Exportable artifacts

> ⚠️ Educational and research use only. Not financial advice.

## 1) Environment Setup
If running on Colab, set **Runtime → Change runtime type → GPU** before training deep models.

In [ ]:
!pip -q install ccxt yfinance pandas numpy scikit-learn matplotlib seaborn ta xgboost lightgbm catboost optuna shap imbalanced-learn tensorflow joblib colorama pytz

## 2) Configuration
Centralized control for data, models, optimization, and exports.

In [ ]:
CONFIG = {
    # Data
    "SYMBOL": "BTC/USDT",
    "EXCHANGE": "binance",
    "TIMEFRAMES": ["1m", "5m", "15m", "1h", "4h", "1d"],
    "PRIMARY_TIMEFRAME": "15m",
    "LOOKBACK_WINDOW": 60,

    # IMPORTANT: maximum candles requested per timeframe fetch call.
    # Increase for longer history, decrease for faster runs.
    "LIMIT_PER_TF": 2000,

    # Optional explicit data window (UTC). Example: "2026-02-01 00:00:00"
    # If None, exchange default recent-history behavior is used.
    "START_DATETIME_UTC": None,
    "END_DATETIME_UTC": None,

    # Labels and split
    "TARGET_THRESHOLD": 0.0,
    "TRAIN_TEST_SPLIT": 0.8,

    # Feature controls
    "N_FEATURES_TARGET": 50,
    "ENABLE_CROSS_TF_FEATURES": True,
    "ENABLE_REGIME_DETECTION": True,

    # Training
    "BATCH_SIZE": 64,
    "EPOCHS_LSTM": 40,
    "PATIENCE": 8,
    "RANDOM_SEED": 42,

    # Optimization
    "ENABLE_OPTUNA": True,
    "OPTUNA_TRIALS": 20,
    "ENABLE_PRUNING": True,

    # Ensemble
    "CONFIDENCE_THRESHOLD": 0.70,
    "META_LEARNER": "LogisticRegression",  # or 'LightGBM'

    # Explainability & export
    "ENABLE_SHAP": True,
    "SAVE_MODELS": True,
    "MODEL_DIR": "/content/models",
    "LOG_DIR": "/content/logs",
    "EXPLAIN_DIR": "/content/explainability",
}

## 3) Imports + IST Logging Utilities

In [ ]:
import os
import json
import random
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import ccxt
import yfinance as yf

from ta.trend import EMAIndicator, SMAIndicator, MACD, ADXIndicator
from ta.momentum import RSIIndicator, StochasticOscillator, WilliamsRIndicator
from ta.volatility import BollingerBands, AverageTrueRange
from ta.volume import OnBalanceVolumeIndicator, MFIIndicator

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.feature_selection import VarianceThreshold, mutual_info_classif
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, matthews_corrcoef, classification_report
from sklearn.linear_model import LogisticRegression

from imblearn.over_sampling import SMOTE

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

import optuna
import shap
import joblib
import pytz
from colorama import Fore, Style, init

init(autoreset=True)

SEED = CONFIG["RANDOM_SEED"]
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

IST = pytz.timezone("Asia/Kolkata")

def log(msg, color=Fore.CYAN):
    ts = datetime.now(IST).strftime("%Y-%m-%d %H:%M:%S %Z")
    print(f"{color}[{ts}] {msg}{Style.RESET_ALL}")

for d in [CONFIG["MODEL_DIR"], CONFIG["LOG_DIR"], CONFIG["EXPLAIN_DIR"]]:
    os.makedirs(d, exist_ok=True)

log("Environment initialized", Fore.GREEN)

## 4) Multi-Timeframe Data Fetching
Primary source: CCXT. Fallback example included for yfinance if needed.

✅ The logic below can force **only fully closed candles** to be used (no currently forming candle leakage).

In [ ]:
def tf_to_timedelta(tf: str) -> pd.Timedelta:
    unit = tf[-1]
    val = int(tf[:-1])
    if unit == "m":
        return pd.Timedelta(minutes=val)
    if unit == "h":
        return pd.Timedelta(hours=val)
    if unit == "d":
        return pd.Timedelta(days=val)
    raise ValueError(f"Unsupported timeframe: {tf}")


def parse_utc(dt_value):
    if dt_value is None:
        return None
    ts = pd.Timestamp(dt_value)
    return ts.tz_localize("UTC") if ts.tzinfo is None else ts.tz_convert("UTC")


def normalize_timeframe_for_exchange(exchange, timeframe: str):
    """
    Return an exchange-supported timeframe or None if unsupported.
    This prevents errors like Coinbase rejecting unsupported granularity values.
    """
    exchange_tfs = getattr(exchange, "timeframes", None)
    if not exchange_tfs:
        # If exchange does not expose timeframe map, use as-is.
        return timeframe

    supported = set(exchange_tfs.keys())
    if timeframe in supported:
        return timeframe

    # Common aliases across exchanges
    aliases = {
        "60m": "1h",
        "1hour": "1h",
        "240m": "4h",
    }
    if timeframe in aliases and aliases[timeframe] in supported:
        return aliases[timeframe]

    return None


def fetch_ohlcv_ccxt(symbol, timeframe, limit=2000, exchange_id="binance", start_utc=None, end_utc=None):
    ex_cls = getattr(ccxt, exchange_id)
    ex = ex_cls({"enableRateLimit": True})

    normalized_tf = normalize_timeframe_for_exchange(ex, timeframe)
    if normalized_tf is None:
        supported = sorted(list(getattr(ex, "timeframes", {}).keys()))
        raise ValueError(
            f"Timeframe '{timeframe}' is not supported on exchange '{exchange_id}'. "
            f"Supported: {supported[:20]}{'...' if len(supported) > 20 else ''}"
        )

    params = {}
    since_ms = None
    if start_utc is not None:
        since_ms = int(pd.Timestamp(start_utc).timestamp() * 1000)

    rows = ex.fetch_ohlcv(symbol, timeframe=normalized_tf, since=since_ms, limit=limit, params=params)
    df = pd.DataFrame(rows, columns=["timestamp", "open", "high", "low", "close", "volume"])
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms", utc=True)
    df = df.sort_values("timestamp").reset_index(drop=True)

    if start_utc is not None:
        df = df[df["timestamp"] >= pd.Timestamp(start_utc)]
    if end_utc is not None:
        df = df[df["timestamp"] <= pd.Timestamp(end_utc)]

    return df.reset_index(drop=True)


def fetch_ohlcv_yfinance(ticker="BTC-USD", interval="15m", period="60d"):
    data = yf.download(ticker, interval=interval, period=period, progress=False, auto_adjust=False)
    data = data.reset_index()
    data.columns = [c.lower().replace(" ", "_") for c in data.columns]
    rename = {"datetime": "timestamp", "date": "timestamp"}
    for k, v in rename.items():
        if k in data.columns:
            data = data.rename(columns={k: v})
    data["timestamp"] = pd.to_datetime(data["timestamp"], utc=True)
    return data[["timestamp", "open", "high", "low", "close", "volume"]].sort_values("timestamp").reset_index(drop=True)


def keep_only_closed_candles(df, timeframe, reference_time_utc=None):
    """
    Keep only candles whose close time is <= last closed boundary.
    Example: now=19:12, tf=15m => keeps candles closed up to 19:00.
             now=19:12, tf=5m  => keeps candles closed up to 19:10.
    """
    if df.empty:
        return df

    now_utc = pd.Timestamp.now(tz="UTC") if reference_time_utc is None else pd.Timestamp(reference_time_utc)
    if now_utc.tzinfo is None:
        now_utc = now_utc.tz_localize("UTC")
    else:
        now_utc = now_utc.tz_convert("UTC")

    delta = tf_to_timedelta(timeframe)
    last_closed_boundary = now_utc.floor(delta)

    close_time = df["timestamp"] + delta
    out = df[close_time <= last_closed_boundary].copy()
    return out.reset_index(drop=True)


def get_multitimeframe_data(config):
    out = {}
    start_utc = parse_utc(config.get("START_DATETIME_UTC"))
    end_utc = parse_utc(config.get("END_DATETIME_UTC"))

    for tf_name in config["TIMEFRAMES"]:
        log(f"Fetching timeframe: {tf_name}")
        try:
            df = fetch_ohlcv_ccxt(
                symbol=config["SYMBOL"],
                timeframe=tf_name,
                limit=config["LIMIT_PER_TF"],
                exchange_id=config["EXCHANGE"],
                start_utc=start_utc,
                end_utc=end_utc,
            )

            # Critical: drop currently forming candle
            df = keep_only_closed_candles(df, tf_name)
            if len(df) == 0:
                log(f"No closed candles returned for {tf_name}; skipping.", Fore.YELLOW)
                continue
            out[tf_name] = df
        except Exception as e:
            log(f"Skipping timeframe {tf_name} due to fetch error: {e}", Fore.YELLOW)

    primary_tf = config["PRIMARY_TIMEFRAME"]
    if primary_tf not in out:
        raise ValueError(
            f"Primary timeframe '{primary_tf}' could not be fetched. "
            f"Available after filtering: {list(out.keys())}. "
            "Choose a supported PRIMARY_TIMEFRAME for your exchange."
        )

    return out

mtf_data = get_multitimeframe_data(CONFIG)
for k, v in mtf_data.items():
    print(k, v.shape, v["timestamp"].min(), "->", v["timestamp"].max())

## 5) Advanced Feature Engineering (100+ target-able features)
Builds per-timeframe features, then merges all into the primary timeframe index.

In [ ]:
def add_base_features(df):
    d = df.copy()

    # returns and volatility
    d["ret_1"] = d["close"].pct_change(1)
    d["ret_5"] = d["close"].pct_change(5)
    d["ret_10"] = d["close"].pct_change(10)
    d["log_ret_1"] = np.log(d["close"] / d["close"].shift(1))

    # candlestick geometry
    d["body"] = d["close"] - d["open"]
    d["body_abs"] = d["body"].abs()
    d["upper_wick"] = d["high"] - d[["open", "close"]].max(axis=1)
    d["lower_wick"] = d[["open", "close"]].min(axis=1) - d["low"]
    d["range"] = d["high"] - d["low"]
    d["body_ratio"] = d["body_abs"] / (d["range"] + 1e-9)

    # simple pattern flags
    d["is_doji"] = (d["body_ratio"] < 0.1).astype(int)
    d["is_hammer"] = ((d["lower_wick"] > 2 * d["body_abs"]) & (d["upper_wick"] < d["body_abs"] + 1e-9)).astype(int)

    # trend indicators
    for w in [7, 14, 21, 50, 100, 200]:
        d[f"sma_{w}"] = SMAIndicator(d["close"], window=w).sma_indicator()
        d[f"ema_{w}"] = EMAIndicator(d["close"], window=w).ema_indicator()

    # momentum
    d["rsi_9"] = RSIIndicator(d["close"], window=9).rsi()
    d["rsi_14"] = RSIIndicator(d["close"], window=14).rsi()
    d["rsi_21"] = RSIIndicator(d["close"], window=21).rsi()

    macd = MACD(d["close"], window_slow=26, window_fast=12, window_sign=9)
    d["macd"] = macd.macd()
    d["macd_signal"] = macd.macd_signal()
    d["macd_diff"] = macd.macd_diff()

    stoch = StochasticOscillator(d["high"], d["low"], d["close"], window=14, smooth_window=3)
    d["stoch_k"] = stoch.stoch()
    d["stoch_d"] = stoch.stoch_signal()
    d["williams_r"] = WilliamsRIndicator(d["high"], d["low"], d["close"], lbp=14).williams_r()

    # volatility and trend strength
    bb = BollingerBands(d["close"], window=20, window_dev=2)
    d["bb_h"] = bb.bollinger_hband()
    d["bb_l"] = bb.bollinger_lband()
    d["bb_m"] = bb.bollinger_mavg()
    d["bb_w"] = bb.bollinger_wband()

    d["atr_14"] = AverageTrueRange(d["high"], d["low"], d["close"], window=14).average_true_range()
    d["atr_21"] = AverageTrueRange(d["high"], d["low"], d["close"], window=21).average_true_range()

    d["adx_14"] = ADXIndicator(d["high"], d["low"], d["close"], window=14).adx()
    d["adx_21"] = ADXIndicator(d["high"], d["low"], d["close"], window=21).adx()

    # volume
    d["obv"] = OnBalanceVolumeIndicator(d["close"], d["volume"]).on_balance_volume()
    d["mfi_14"] = MFIIndicator(d["high"], d["low"], d["close"], d["volume"], window=14).money_flow_index()

    # statistical features
    for w in [20, 50]:
        roll_mean = d["close"].rolling(w).mean()
        roll_std = d["close"].rolling(w).std()
        d[f"zscore_{w}"] = (d["close"] - roll_mean) / (roll_std + 1e-9)

    for lag in [1, 2, 3, 5, 10, 20]:
        d[f"close_lag_{lag}"] = d["close"].shift(lag)
        d[f"volume_lag_{lag}"] = d["volume"].shift(lag)

    # cyclical time features
    d["hour"] = d["timestamp"].dt.hour
    d["dow"] = d["timestamp"].dt.dayofweek
    d["hour_sin"] = np.sin(2 * np.pi * d["hour"] / 24)
    d["hour_cos"] = np.cos(2 * np.pi * d["hour"] / 24)
    d["dow_sin"] = np.sin(2 * np.pi * d["dow"] / 7)
    d["dow_cos"] = np.cos(2 * np.pi * d["dow"] / 7)

    return d


def prefix_non_timestamp_cols(df, prefix):
    cols = {c: f"{prefix}_{c}" for c in df.columns if c != "timestamp"}
    return df.rename(columns=cols)


def build_feature_table(mtf_data, config):
    primary_tf = config["PRIMARY_TIMEFRAME"]

    feat_tables = {}
    for tf_name, tf_df in mtf_data.items():
        f = add_base_features(tf_df)
        feat_tables[tf_name] = prefix_non_timestamp_cols(f, tf_name)

    base = feat_tables[primary_tf].copy().sort_values("timestamp")

    # align other timeframes to primary using backward asof join
    for tf_name, tf_feat in feat_tables.items():
        if tf_name == primary_tf:
            continue
        right = tf_feat.sort_values("timestamp")
        base = pd.merge_asof(
            base.sort_values("timestamp"),
            right.sort_values("timestamp"),
            on="timestamp",
            direction="backward",
        )

    return base

feature_df = build_feature_table(mtf_data, CONFIG)
print("feature table shape:", feature_df.shape)
feature_df.head(2)

## 6) Target Generation + Feature Selection
- Label: next-candle up/down based on primary timeframe close return.
- Selection: variance threshold + mutual information top-N.

In [ ]:
def add_target_and_clean(feature_df, config):
    d = feature_df.copy()
    pfx = config["PRIMARY_TIMEFRAME"]
    close_col = f"{pfx}_close"

    d["target_return_next"] = d[close_col].shift(-1) / d[close_col] - 1
    d["target"] = (d["target_return_next"] > config["TARGET_THRESHOLD"]).astype(int)

    d = d.replace([np.inf, -np.inf], np.nan)
    d = d.ffill().bfill().dropna().reset_index(drop=True)
    return d


def select_features(df, config):
    ignore_cols = {"timestamp", "target", "target_return_next"}
    all_features = [c for c in df.columns if c not in ignore_cols]

    X = df[all_features]
    y = df["target"]

    vt = VarianceThreshold(threshold=0.0)
    X_vt = vt.fit_transform(X)
    vt_cols = X.columns[vt.get_support()]
    X_vt_df = pd.DataFrame(X_vt, columns=vt_cols, index=X.index)

    mi = mutual_info_classif(X_vt_df.fillna(0), y, random_state=SEED)
    mi_s = pd.Series(mi, index=X_vt_df.columns).sort_values(ascending=False)
    selected = mi_s.head(config["N_FEATURES_TARGET"]).index.tolist()

    return selected, mi_s

model_df = add_target_and_clean(feature_df, CONFIG)
selected_features, mi_scores = select_features(model_df, CONFIG)

print("Model DF:", model_df.shape)
print("Selected features:", len(selected_features))
mi_scores.head(10)

## 7) Chronological Train/Test Split

In [ ]:
def chronological_split(df, feature_cols, split_ratio=0.8):
    split_idx = int(len(df) * split_ratio)
    X = df[feature_cols]
    y = df["target"]

    X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
    y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
    ts_train = df.iloc[:split_idx]["timestamp"]
    ts_test = df.iloc[split_idx:]["timestamp"]
    return X_train, X_test, y_train, y_test, ts_train, ts_test

X_train_raw, X_test_raw, y_train_raw, y_test, ts_train, ts_test = chronological_split(
    model_df, selected_features, CONFIG["TRAIN_TEST_SPLIT"]
)

scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw)

X_train = pd.DataFrame(X_train_scaled, columns=selected_features, index=X_train_raw.index)
X_test = pd.DataFrame(X_test_scaled, columns=selected_features, index=X_test_raw.index)

print("Train/Test shapes:", X_train.shape, X_test.shape)

## 8) Class Balancing (SMOTE)
Applied only to train split.

In [ ]:
smote = SMOTE(random_state=SEED)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train_raw)

print("Before SMOTE:", y_train_raw.value_counts().to_dict())
print("After SMOTE:", pd.Series(y_train_bal).value_counts().to_dict())

## 9) Sequence Creation for Deep Learning

In [ ]:
def create_sequences(X_df, y_series, lookback=60):
    Xv = X_df.values
    yv = y_series.values

    X_seq, y_seq = [], []
    for i in range(lookback, len(Xv)):
        X_seq.append(Xv[i - lookback:i])
        y_seq.append(yv[i])

    return np.array(X_seq, dtype=np.float32), np.array(y_seq, dtype=np.float32)

lookback = CONFIG["LOOKBACK_WINDOW"]
X_train_seq, y_train_seq = create_sequences(X_train, y_train_raw, lookback)
X_test_seq, y_test_seq = create_sequences(X_test, y_test, lookback)

print("Sequence train/test:", X_train_seq.shape, X_test_seq.shape)

## 10) Tree-Based Models (XGBoost, LightGBM, CatBoost) + Optuna

In [ ]:
def evaluate_binary(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
        "auc": roc_auc_score(y_true, y_prob),
        "mcc": matthews_corrcoef(y_true, y_pred),
    }


def optuna_objective_xgb(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "random_state": SEED,
        "eval_metric": "logloss",
    }
    model = XGBClassifier(**params)
    model.fit(X_train_bal, y_train_bal)
    prob = model.predict_proba(X_test)[:, 1]
    return f1_score(y_test, (prob >= 0.5).astype(int))


if CONFIG["ENABLE_OPTUNA"]:
    study = optuna.create_study(direction="maximize")
    study.optimize(optuna_objective_xgb, n_trials=CONFIG["OPTUNA_TRIALS"])
    best_xgb_params = study.best_params
    log(f"Best XGB F1 from Optuna: {study.best_value:.4f}", Fore.GREEN)
else:
    best_xgb_params = {
        "n_estimators": 300,
        "max_depth": 6,
        "learning_rate": 0.05,
        "subsample": 0.9,
        "colsample_bytree": 0.9,
        "min_child_weight": 3,
        "reg_alpha": 1e-4,
        "reg_lambda": 1.0,
    }

xgb_model = XGBClassifier(**best_xgb_params, random_state=SEED, eval_metric="logloss")
lgb_model = LGBMClassifier(n_estimators=400, learning_rate=0.03, max_depth=-1, random_state=SEED)
cat_model = CatBoostClassifier(iterations=400, learning_rate=0.03, depth=8, random_seed=SEED, verbose=0)

xgb_model.fit(X_train_bal, y_train_bal)
lgb_model.fit(X_train_bal, y_train_bal)
cat_model.fit(X_train_bal, y_train_bal)

xgb_prob = xgb_model.predict_proba(X_test)[:, 1]
lgb_prob = lgb_model.predict_proba(X_test)[:, 1]
cat_prob = cat_model.predict_proba(X_test)[:, 1]

print("XGB:", evaluate_binary(y_test, xgb_prob))
print("LGBM:", evaluate_binary(y_test, lgb_prob))
print("CAT:", evaluate_binary(y_test, cat_prob))

## 11) Deep Learning Models (Attention-BiLSTM + CNN-LSTM)

In [ ]:
def build_attention_bilstm(input_shape):
    inp = layers.Input(shape=input_shape)
    x = layers.Bidirectional(layers.LSTM(128, return_sequences=True))(inp)
    x = layers.Dropout(0.3)(x)
    x = layers.Bidirectional(layers.LSTM(64, return_sequences=True))(x)

    # attention weights
    attn = layers.Dense(1, activation="tanh")(x)
    attn = layers.Flatten()(attn)
    attn = layers.Activation("softmax")(attn)
    attn = layers.RepeatVector(128)(attn)
    attn = layers.Permute([2, 1])(attn)

    x = layers.Multiply()([x, attn])
    x = layers.Lambda(lambda t: tf.reduce_sum(t, axis=1))(x)

    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(1, activation="sigmoid")(x)

    model = models.Model(inp, out)
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])
    return model


def build_cnn_lstm(input_shape):
    inp = layers.Input(shape=input_shape)
    x = layers.Conv1D(64, kernel_size=3, activation="relu", padding="same")(inp)
    x = layers.MaxPooling1D(pool_size=2)(x)
    x = layers.Conv1D(128, kernel_size=3, activation="relu", padding="same")(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
    x = layers.LSTM(100, return_sequences=True)(x)
    x = layers.LSTM(50)(x)
    x = layers.Dense(50, activation="relu")(x)
    out = layers.Dense(1, activation="sigmoid")(x)

    model = models.Model(inp, out)
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])
    return model

es = callbacks.EarlyStopping(monitor="val_loss", patience=CONFIG["PATIENCE"], restore_best_weights=True)

attn_model = build_attention_bilstm((X_train_seq.shape[1], X_train_seq.shape[2]))
cnn_lstm_model = build_cnn_lstm((X_train_seq.shape[1], X_train_seq.shape[2]))

hist_attn = attn_model.fit(
    X_train_seq, y_train_seq,
    validation_split=0.2,
    epochs=CONFIG["EPOCHS_LSTM"],
    batch_size=CONFIG["BATCH_SIZE"],
    callbacks=[es],
    verbose=1,
)

hist_cnn = cnn_lstm_model.fit(
    X_train_seq, y_train_seq,
    validation_split=0.2,
    epochs=CONFIG["EPOCHS_LSTM"],
    batch_size=CONFIG["BATCH_SIZE"],
    callbacks=[es],
    verbose=1,
)

attn_prob = attn_model.predict(X_test_seq, verbose=0).ravel()
cnn_prob = cnn_lstm_model.predict(X_test_seq, verbose=0).ravel()

print("Attention-BiLSTM:", evaluate_binary(y_test_seq, attn_prob))
print("CNN-LSTM:", evaluate_binary(y_test_seq, cnn_prob))

## 12) Hierarchical Ensemble (Stacking Meta-Learner)

In [ ]:
# align tree preds to sequence test index length
trim = len(y_test_seq)
xgb_prob_trim = xgb_prob[-trim:]
lgb_prob_trim = lgb_prob[-trim:]
cat_prob_trim = cat_prob[-trim:]

meta_X = np.column_stack([xgb_prob_trim, lgb_prob_trim, cat_prob_trim, attn_prob, cnn_prob])
meta_y = y_test_seq.astype(int)

if CONFIG["META_LEARNER"] == "LightGBM":
    meta_model = LGBMClassifier(n_estimators=200, learning_rate=0.03, random_state=SEED)
else:
    meta_model = LogisticRegression(max_iter=2000, random_state=SEED)

meta_model.fit(meta_X, meta_y)
meta_prob = meta_model.predict_proba(meta_X)[:, 1]
meta_pred = (meta_prob >= 0.5).astype(int)

metrics = {
    "accuracy": accuracy_score(meta_y, meta_pred),
    "f1": f1_score(meta_y, meta_pred),
    "auc": roc_auc_score(meta_y, meta_prob),
    "mcc": matthews_corrcoef(meta_y, meta_pred),
}

print("Ensemble metrics:", metrics)
print(classification_report(meta_y, meta_pred, digits=4))

## 13) SHAP Explainability
SHAP is computed on one tree model (XGBoost) for speed and clarity.

In [ ]:
if CONFIG["ENABLE_SHAP"]:
    explainer = shap.TreeExplainer(xgb_model)
    sample_X = X_test.sample(min(500, len(X_test)), random_state=SEED)
    shap_values = explainer.shap_values(sample_X)

    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values, sample_X, show=False)
    shap_path = os.path.join(CONFIG["EXPLAIN_DIR"], "shap_summary.png")
    plt.tight_layout()
    plt.savefig(shap_path, dpi=150)
    plt.show()

    np.save(os.path.join(CONFIG["EXPLAIN_DIR"], "shap_values.npy"), np.array(shap_values))
    log(f"Saved SHAP artifacts in {CONFIG['EXPLAIN_DIR']}", Fore.GREEN)
else:
    log("SHAP disabled in config", Fore.YELLOW)

## 14) Model Export

In [ ]:
if CONFIG["SAVE_MODELS"]:
    tree_dir = os.path.join(CONFIG["MODEL_DIR"], "tree_models")
    dl_dir = os.path.join(CONFIG["MODEL_DIR"], "dl_models")
    meta_dir = os.path.join(CONFIG["MODEL_DIR"], "meta")

    os.makedirs(tree_dir, exist_ok=True)
    os.makedirs(dl_dir, exist_ok=True)
    os.makedirs(meta_dir, exist_ok=True)

    joblib.dump(xgb_model, os.path.join(tree_dir, "xgboost.pkl"))
    joblib.dump(lgb_model, os.path.join(tree_dir, "lightgbm.pkl"))
    joblib.dump(cat_model, os.path.join(tree_dir, "catboost.pkl"))

    attn_model.save(os.path.join(dl_dir, "attention_bilstm.h5"))
    cnn_lstm_model.save(os.path.join(dl_dir, "cnn_lstm.h5"))

    joblib.dump(meta_model, os.path.join(meta_dir, "meta_learner.pkl"))
    joblib.dump(scaler, os.path.join(CONFIG["MODEL_DIR"], "scaler.pkl"))

    with open(os.path.join(CONFIG["MODEL_DIR"], "feature_names.json"), "w") as f:
        json.dump(selected_features, f, indent=2)

    with open(os.path.join(CONFIG["MODEL_DIR"], "config.json"), "w") as f:
        json.dump(CONFIG, f, indent=2)

    metadata = {
        "created_at_ist": datetime.now(IST).isoformat(),
        "metrics": metrics,
        "n_selected_features": len(selected_features),
        "lookback_window": CONFIG["LOOKBACK_WINDOW"],
    }
    with open(os.path.join(CONFIG["MODEL_DIR"], "metadata.json"), "w") as f:
        json.dump(metadata, f, indent=2)

    log(f"Models exported to {CONFIG['MODEL_DIR']}", Fore.GREEN)
else:
    log("Model saving disabled", Fore.YELLOW)

## 15) Final Summary + Expected Performance Bands

In [ ]:
high_conf_mask = meta_prob >= CONFIG["CONFIDENCE_THRESHOLD"]
if high_conf_mask.sum() > 0:
    high_acc = accuracy_score(meta_y[high_conf_mask], meta_pred[high_conf_mask])
    high_f1 = f1_score(meta_y[high_conf_mask], meta_pred[high_conf_mask])
else:
    high_acc, high_f1 = np.nan, np.nan

summary = pd.DataFrame([
    {
        "Metric": "Accuracy",
        "Overall": metrics["accuracy"],
        "High-Confidence": high_acc,
        "Target Range": "0.78 - 0.84"
    },
    {
        "Metric": "F1",
        "Overall": metrics["f1"],
        "High-Confidence": high_f1,
        "Target Range": "0.83 - 0.87"
    },
    {
        "Metric": "AUC",
        "Overall": metrics["auc"],
        "High-Confidence": np.nan,
        "Target Range": "0.87 - 0.91"
    },
    {
        "Metric": "MCC",
        "Overall": metrics["mcc"],
        "High-Confidence": np.nan,
        "Target Range": "0.65 - 0.75"
    },
])

print(summary)

## 16) Real-Time Usage Example
Use exported models for inference on fresh feature data.

In [ ]:
def predict_next_candle(features_df_latest_window, exported_dir="/content/models"):
    xgb_m = joblib.load(f"{exported_dir}/tree_models/xgboost.pkl")
    lgb_m = joblib.load(f"{exported_dir}/tree_models/lightgbm.pkl")
    cat_m = joblib.load(f"{exported_dir}/tree_models/catboost.pkl")
    meta_m = joblib.load(f"{exported_dir}/meta/meta_learner.pkl")
    scaler_m = joblib.load(f"{exported_dir}/scaler.pkl")

    with open(f"{exported_dir}/feature_names.json", "r") as f:
        feats = json.load(f)

    # deep models are optional for live use if lookback rows are available
    attn_m = tf.keras.models.load_model(f"{exported_dir}/dl_models/attention_bilstm.h5")
    cnn_m = tf.keras.models.load_model(f"{exported_dir}/dl_models/cnn_lstm.h5")

    Xf = features_df_latest_window[feats].copy()
    Xs = scaler_m.transform(Xf)

    xgb_p = xgb_m.predict_proba(Xs[-1:])[:, 1][0]
    lgb_p = lgb_m.predict_proba(Xs[-1:])[:, 1][0]
    cat_p = cat_m.predict_proba(Xs[-1:])[:, 1][0]

    lookback = CONFIG["LOOKBACK_WINDOW"]
    if len(Xs) >= lookback:
        seq = Xs[-lookback:].reshape(1, lookback, -1)
        attn_p = float(attn_m.predict(seq, verbose=0)[0][0])
        cnn_p = float(cnn_m.predict(seq, verbose=0)[0][0])
    else:
        attn_p, cnn_p = 0.5, 0.5

    meta_in = np.array([[xgb_p, lgb_p, cat_p, attn_p, cnn_p]])
    prob_up = float(meta_m.predict_proba(meta_in)[:, 1][0])
    pred_up = int(prob_up >= 0.5)

    return pred_up, prob_up

print("Inference function ready. Pass latest engineered feature rows to predict_next_candle(...)")

---
## Best Practices Checklist
- Retrain periodically (weekly/monthly).
- Monitor drift and live performance.
- Use position sizing and strict risk management.
- Backtest with fees/slippage and realistic execution rules.
- Treat this as decision support, not certainty.

## 17) 🔴 Real Next-Candle Prediction (Live, Closed-Candle Safe)
This cell makes real-time inference using only data up to the **last fully closed candle**.

Examples:
- If current time is 19:12 and timeframe is 15m → last closed boundary is 19:00.
- If current time is 19:12 and timeframe is 5m  → last closed boundary is 19:10.

So the model predicts the **next candle**, never the currently forming candle.

In [ ]:
# Live prediction using latest market data (primary timeframe), excluding forming candle
live_df = fetch_ohlcv_ccxt(
    symbol=CONFIG["SYMBOL"],
    timeframe=CONFIG["PRIMARY_TIMEFRAME"],
    limit=max(500, CONFIG["LOOKBACK_WINDOW"] + 120),
    exchange_id=CONFIG["EXCHANGE"],
    start_utc=parse_utc(CONFIG.get("START_DATETIME_UTC")),
    end_utc=parse_utc(CONFIG.get("END_DATETIME_UTC")),
)

live_df = keep_only_closed_candles(live_df, CONFIG["PRIMARY_TIMEFRAME"])
if len(live_df) < (CONFIG["LOOKBACK_WINDOW"] + 5):
    raise ValueError("Not enough closed candles after filtering. Increase LIMIT_PER_TF or widen start/end range.")

live_feat = add_base_features(live_df).replace([np.inf, -np.inf], np.nan).ffill().bfill().dropna().reset_index(drop=True)

# Build inference matrix with selected training features only
X_live = live_feat[selected_features].copy()
X_live_scaled = scaler.transform(X_live)

# Tree model probabilities for latest closed candle
xgb_live = float(xgb_model.predict_proba(X_live_scaled[-1:])[:, 1][0])
lgb_live = float(lgb_model.predict_proba(X_live_scaled[-1:])[:, 1][0])
cat_live = float(cat_model.predict_proba(X_live_scaled[-1:])[:, 1][0])

# Deep model probabilities (if enough lookback)
if len(X_live_scaled) >= CONFIG["LOOKBACK_WINDOW"]:
    seq = X_live_scaled[-CONFIG["LOOKBACK_WINDOW"]:].reshape(1, CONFIG["LOOKBACK_WINDOW"], -1)
    attn_live = float(attn_model.predict(seq, verbose=0)[0][0])
    cnn_live = float(cnn_lstm_model.predict(seq, verbose=0)[0][0])
else:
    attn_live, cnn_live = 0.5, 0.5

meta_input_live = np.array([[xgb_live, lgb_live, cat_live, attn_live, cnn_live]])
prob_up_live = float(meta_model.predict_proba(meta_input_live)[:, 1][0])
pred_live = int(prob_up_live >= 0.5)

delta_live = tf_to_timedelta(CONFIG["PRIMARY_TIMEFRAME"])
last_open_ts = live_feat["timestamp"].iloc[-1]
last_close_ts = last_open_ts + delta_live
next_open_ts = last_close_ts
next_close_ts = next_open_ts + delta_live
last_close_price = float(live_feat["close"].iloc[-1])

print("=== LIVE NEXT-CANDLE PREDICTION (CLOSED-CANDLE SAFE) ===")
print(f"Symbol / TF: {CONFIG['SYMBOL']} / {CONFIG['PRIMARY_TIMEFRAME']}")
print(f"Last CLOSED candle window (UTC): {last_open_ts} -> {last_close_ts}")
print(f"Latest close used for inference: {last_close_price:.2f}")
print(f"Predicting NEXT candle window (UTC): {next_open_ts} -> {next_close_ts}")
print(f"Predicted next candle direction: {'UP' if pred_live == 1 else 'DOWN'}")
print(f"Probability UP: {prob_up_live:.4f} | Probability DOWN: {1-prob_up_live:.4f}")
print(f"Confidence score: {max(prob_up_live, 1-prob_up_live):.4f}")